# QCAF Quantum Admissibility Experiment

Reproducible Colab example for bounded quantum admissibility using MiniZinc.

INADMISSIBLE applies only to the supplied observation set and thresholds; it is not a proof of global impossibility.


In [ ]:
# @title Install QCAF and scientific dependencies
!apt-get -qq update >/dev/null
!apt-get -qq install -y minizinc >/dev/null
%pip -q install git+https://github.com/MarceloClaro/minizinc-mcp.git pennylane qiskit qiskit-aer
print("QCAF environment ready.")


In [ ]:
# @title Load QCAF
import json
from pprint import pprint
from main import quantum_admissibility_core
from qcaf import QuantumAdmissibilityRequest


In [ ]:
# @title Reproducible bounded request
request_payload = json.loads("{\n  \"observations\": [\n    {\n      \"label\": \"pennylane-run-001\",\n      \"framework\": \"pennylane\",\n      \"backend\": \"default.mixed\",\n      \"target_probability\": 0.94,\n      \"ber\": 0.03,\n      \"noise_probability\": 0.04,\n      \"circuit_depth\": 12,\n      \"shots\": 8192,\n      \"fidelity\": 0.97,\n      \"seed\": 42,\n      \"parameters\": {\n        \"theta\": [\n          0.3,\n          0.7\n        ]\n      },\n      \"metadata\": {\n        \"purpose\": \"QCAF reproducibility example\"\n      }\n    },\n    {\n      \"label\": \"qiskit-run-002\",\n      \"framework\": \"qiskit\",\n      \"backend\": \"aer_simulator\",\n      \"target_probability\": 0.88,\n      \"ber\": 0.06,\n      \"noise_probability\": 0.08,\n      \"circuit_depth\": 18,\n      \"shots\": 8192,\n      \"seed\": 42\n    }\n  ],\n  \"constraints\": {\n    \"min_target_probability\": 0.9,\n    \"max_ber\": 0.05,\n    \"max_noise_probability\": 0.1,\n    \"max_circuit_depth\": 20,\n    \"min_shots\": 8192,\n    \"min_fidelity\": 0.95\n  },\n  \"objective\": \"maximize_target_probability\",\n  \"solver\": \"gecode\",\n  \"timeout\": 10\n}")
qcaf_request = QuantumAdmissibilityRequest(**request_payload)
pprint(request_payload)


## PennyLane / Qiskit integration

Replace or extend the embedded observations with measurements generated from your declared PennyLane or Qiskit parameter/noise grid. Preserve seeds, circuit hashes, versions and backend metadata for reproducibility.


In [ ]:
# @title Solve QCAF
result = await quantum_admissibility_core(qcaf_request)
result_data = result.model_dump() if hasattr(result, "model_dump") else result.dict()
pprint(result_data)


In [ ]:
# @title Interpretation
print("Admissibility:", result.admissibility)
print("Solver status:", result.solver_status)
print("Objective:", result.objective)
print("Interpretation boundary:")
print(result.interpretation_boundary)
if result.selected_observation is not None:
    print("\nSelected witness:")
    selected = result.selected_observation.model_dump() if hasattr(result.selected_observation, "model_dump") else result.selected_observation.dict()
    pprint(selected)
